# Lab4a Named-entity-recognition using fine-tuned transformers

Copyright: Vrije Universiteit Amsterdam, Faculty of Humanities, CLTL

Before reading this notebook make sure you have consulted **Lab3.4 SentimentClassification using transformer models**, which contains some disclaimers, tips and explains the sentence representations obtained from the transformer models.

In this notebook we will use the simpletransformer package that provides a simple API on top of the transformer packge.

In [ ]:
#Requires installing transformers, pytorch and simpletransformers
#!conda install pytorch cpuonly -c pytorch
#!pip install transformers
#!pip install simpletransformers

We load a transformer model 'bert-base-NER' from the Hugging face repository, which is fine-tuned for Named Entity recognition: 

https://huggingface.co/models

We need to load the model for the sequence classifcation and the tokenizer to convert the sentences into tokens according to the vocabulary of the model.

Loading the model takes some time and requires you have sufficient memory to load the model

In [1]:
from simpletransformers.ner import NERModel
#sentences = ["Example sentence 1", "Example sentence 2"]
englishmodel = NERModel(
        model_type="bert",
        model_name="dslim/bert-base-NER",
        use_cuda=False
)

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

We create an instance of the NERModel that can be used for training, evaluation, and prediction in Named-Entity-Recognition (NER) tasks. The full parameter list for a NERModel object:

* model_type: The type of model (bert, roberta)
* model_name: Default Transformer model name or path to a directory containing Transformer model file (pytorch_nodel.bin).
* labels (optional): A list of all Named Entity labels. If not given, [“O”, “B-MISC”, “I-MISC”, “B-PER”, “I-PER”, “B-ORG”, “I-ORG”, “B-LOC”, “I-LOC”] will be used.
* args (optional): Default args will be used if this parameter is not provided. If provided, it should be a dict containing the args that should be changed in the default args.
* use_cuda (optional): Use GPU if available. Setting to False will force model to use CPU only.

In [2]:
predictions, raw_outputs = englishmodel.predict(["Apple sued Samsung for patents last year."])

  0%|          | 0/1 [00:00<?, ?it/s]

Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
predictions

[[{'Apple': 'B-ORG'},
  {'sued': 'O'},
  {'Samsung': 'B-ORG'},
  {'for': 'O'},
  {'patents': 'O'},
  {'last': 'O'},
  {'year.': 'O'}]]

In [4]:
dutchmodel = NERModel(
        model_type="bert",
        model_name="Matthijsvanhof/bert-base-dutch-cased-finetuned-NER",
        use_cuda=False
)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/434M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/434M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
predictions, raw_outputs = dutchmodel.predict(["Apple sleept Samsung voor de rechter vanwege schending van patenten."])

  0%|          | 0/1 [00:00<?, ?it/s]

Running Prediction:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
predictions

[[{'Apple': 'O'},
  {'sleept': 'O'},
  {'Samsung': 'B-MISC'},
  {'voor': 'O'},
  {'de': 'O'},
  {'rechter': 'O'},
  {'vanwege': 'O'},
  {'schending': 'O'},
  {'van': 'O'},
  {'patenten.': 'O'}]]

Another option for Dutch NER (https://huggingface.co/flair/ner-dutch-large):

In [2]:
from flair.data import Sentence
from flair.models import SequenceTagger

# load tagger
tagger = SequenceTagger.load("flair/ner-dutch-large")

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2026-05-10 18:22:32,151 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-PER, S-LOC, B-MISC, E-MISC, B-ORG, E-ORG, I-ORG, I-PER, B-LOC, I-LOC, E-LOC, I-MISC, <START>, <STOP>


In [3]:
sentence = Sentence("Apple sleept Samsung voor de rechter vanwege schending van patenten.")

# predict NER tags
tagger.predict(sentence)

# print sentence
print(sentence)

# print predicted NER spans
print('The following NER tags are found:')
# iterate over entities and print
for entity in sentence.get_spans('ner'):
    print(entity)

Sentence[11]: "Apple sleept Samsung voor de rechter vanwege schending van patenten." → ["Apple"/ORG, "Samsung"/ORG]
The following NER tags are found:
Span[0:1]: "Apple" → ORG (1.0000)
Span[2:3]: "Samsung" → ORG (1.0000)


# End of this notebook